## This jupyter notebook gets all the necessary Anderson Metadata into the proper format to incoorporate in the model building.
Results in anderson_metadata.csv

In [3]:
import pandas as pd
import re

In [4]:
%cd ..

/proj/gibbons/2025_methane_Andersen


/users/jschoch/miniconda3/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
exp_design_summary = pd.read_csv("table_s1_experimental_design_summary.csv")
exp_design_summary

,Animal,Identification,Period 1,Period 2,Period 3,Period 4
0,Cow,2011612,HPO,MAP,CTL,COS
1,Cow,2012046,MAP,COS,HPO,CTL
2,Cow,2012175,CTL,HPO,COS,MAP
3,Cow,2012179,COS,CTL,MAP,HPO
4,Goat,13035,MAP,COS,HPO,CTL
5,Goat,14100,CTL,HPO,COS,MAP
6,Goat,14188,HPO,MAP,CTL,COS
7,Goat,14201,COS,CTL,MAP,HPO


In [6]:
# Melt the diet DataFrame to long format for easier lookup
diet_long = pd.melt(
    exp_design_summary,
    id_vars=['Animal', 'Identification'],
    value_vars=['Period 1', 'Period 2', 'Period 3', 'Period 4'],
    var_name='Period',
    value_name='Diet'
)

# Extract just the period number for consistency
diet_long['Period'] = diet_long['Period'].str.extract(r'(\d+)').astype(int)


diet_long['Identification'] = diet_long['Identification'].astype(str).str.replace(r'^201', '', regex=True)
diet_long['Identification'] = pd.to_numeric(diet_long['Identification'], errors='coerce')
diet_long = diet_long.sort_values(by=['Identification', 'Period']).reset_index(drop=True)

diet_long.index = diet_long.index + 1


# Preview
#diet_long


In [7]:
sra_info = pd.read_csv("SraRunInfo.csv")
sample_mapping = sra_info[["Run", "SampleName","Submission"]]
sample_mapping.index = sample_mapping["SampleName"].str.split('_').str[0].astype(int)
sample_mapping.index.name = None

In [18]:
metadata = diet_long.merge(sample_mapping, left_index=True, right_index=True, how='inner')

In [28]:
metadata = metadata[["Animal", "Identification", "Period", "Diet", "Run"]]
metadata = metadata.rename(columns={"Run": "sample_id"})

In [30]:
metadata.to_csv("workstation/anderson_metadata.csv", index = False)

In [ ]:
pd.read_csv("anderson_metadata.csv")

In [29]:
metadata

,Animal,Identification,Period,Diet,sample_id
1,Cow,1612,1,HPO,SRR19524268
2,Cow,1612,2,MAP,SRR19524256
3,Cow,1612,3,CTL,SRR19524252
4,Cow,1612,4,COS,SRR19524251
5,Cow,2046,1,MAP,SRR19524250
6,Cow,2046,2,COS,SRR19524249
7,Cow,2046,3,HPO,SRR19524248
8,Cow,2046,4,CTL,SRR19524246
9,Cow,2175,1,CTL,SRR19524245
10,Cow,2175,2,HPO,SRR19524270
